In [12]:
import sys
import csv
import os
import json
import torch
import pandas as pd
from dotenv import load_dotenv, find_dotenv

# Load .env
load_dotenv(find_dotenv())

# -------------------------
# Config
# -------------------------
SEED = int(os.environ.get("SEED", 42))
DATA_DIR = os.environ.get("DATA_DIR", "/workspace/data")

parsed_docs_dir = os.path.join(DATA_DIR, "parsed_docs")
train_csv_path = os.path.join(parsed_docs_dir, "training_final.csv")
test_csv_path = os.path.join(parsed_docs_dir, "test_final.csv")

# -------------------------
# Environment Info
# -------------------------
print(f"SEED: {SEED}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"PyTorch version: {torch.__version__}")
print(f"Data dir: {DATA_DIR}")
print(f"Train CSV path: {train_csv_path}")
print(f"Test CSV path: {test_csv_path}")

SEED: 2026
CUDA available: True
PyTorch version: 2.2.0
Data dir: /workspace/data
Train CSV path: /workspace/data/parsed_docs/training_final.csv
Test CSV path: /workspace/data/parsed_docs/test_final.csv


## Debugging

In [15]:
with open(train_csv_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if line.count('"') % 2 != 0:
            print(f"⚠️ Possible broken quotes at line {i}")
            print(line[:500])

⚠️ Possible broken quotes at line 77
What is the method FASP used for?,"PURPOSE: The aim of this study was to elucidate the role of lysyl oxidase (LOX) and lysyl oxidase like (LOXL) 2 in pathologic wound healing after glaucoma surgery. We therefore investigated the expression of LOX and LOXL2 and evaluated the therapeutic potential of anti-LOX (GS-639556, formerly M64) and anti-LOXL2 (GS-607601, formerly AB0023) antibodies in a rabbit model of glaucoma trabeculectomy. METHODS: Ocular expression of LOX and LOXL2 was investigated by 


## Load the CSV and parse the abstracts column

In [36]:
# Increase CSV field limit
csv.field_size_limit(sys.maxsize)

# Read only first 10 rows
df = pd.read_csv(train_csv_path, engine="python", nrows=10)

# Convert JSON string -> Python list
df["abstracts"] = df["abstracts"].apply(json.loads)

In [37]:
df

,question,type,text,category,topic_id,confidence,n_sources,source_urls,pmids,abstracts
0,Is Hirschsprung disease a mendelian or a multi...,summary,OBJECTIVE: To analyze the prognostic factors o...,Surgery & Procedures,9,0.3225,16,"http://www.ncbi.nlm.nih.gov/pubmed/15829955, h...","15829955, 15617541, 12239580, 12239580, 158299...","[{'pmid': '15829955', 'text': 'OBJECTIVE: To a..."
1,List signaling molecules (ligands) that intera...,list,PURPOSE: Histone deacetylase (HDAC) inhibitors...,Pharmacology & Drugs,2,0.1407,15,"http://www.ncbi.nlm.nih.gov/pubmed/24323361, h...","24323361, 24124521, 23959273, 23888072, 238213...","[{'pmid': '24323361', 'text': 'PURPOSE: Histon..."
2,Is the protein Papilin secreted?,yesno,INTRODUCTION: Insertion of a ventriculoperiton...,Infectious Disease,5,0.2186,8,"http://www.ncbi.nlm.nih.gov/pubmed/21784067, h...","21784067, 19297413, 15094122, 15094110, 126662...","[{'pmid': '21784067', 'text': 'INTRODUCTION: I..."
3,Are long non coding RNAs spliced?,yesno,"BACKGROUND: Imatinib mesylate, an orally admin...",Molecular Biology,12,0.3204,6,"http://www.ncbi.nlm.nih.gov/pubmed/22955988, h...","22955988, 22955974, 22707570, 21622663, 242853...","[{'pmid': '22955988', 'text': 'BACKGROUND: Ima..."
4,Is RANKL secreted from the cells?,yesno,BACKGROUND: Chronic cerebrospinal venous insuf...,Surgery & Procedures,9,0.2285,11,"http://www.ncbi.nlm.nih.gov/pubmed/24267510, h...","24267510, 24265865, 23835909, 23827649, 236987...","[{'pmid': '24267510', 'text': 'BACKGROUND: Chr..."
5,Does metformin interfere thyroxine absorption?,yesno,BACKGROUND: Colorectal cancer (CRC) is one of ...,Genetics & Mutations,0,0.3416,2,"http://www.ncbi.nlm.nih.gov/pubmed/26191653, h...","26191653, 26191653","[{'pmid': '26191653', 'text': 'BACKGROUND: Col..."
6,Which miRNAs could be used as potential biomar...,list,Background: The scratch collapse test (SCT) is...,Surgery & Procedures,9,0.2328,22,"http://www.ncbi.nlm.nih.gov/pubmed/23978303, h...","23978303, 23918241, 23918241, 23888941, 238889...","[{'pmid': '23978303', 'text': 'Background: The..."
7,Which acetylcholinesterase inhibitors are used...,list,OBJECTIVE: To examine reproductive history and...,Other,14,0.1622,11,"http://www.ncbi.nlm.nih.gov/pubmed/21133188, h...","21133188, 20663605, 20663605, 20663605, 218157...","[{'pmid': '21133188', 'text': 'OBJECTIVE: To e..."
8,Has Denosumab (Prolia) been approved by FDA?,yesno,PURPOSE: To present the serial imaging finding...,Surgery & Procedures,9,0.5328,22,"http://www.ncbi.nlm.nih.gov/pubmed/24316116, h...","24316116, 24316116, 24308016, 24126422, 241146...","[{'pmid': '24316116', 'text': 'PURPOSE: To pre..."
9,List the human genes encoding for the dishevel...,list,BACKGROUND: Neural tube defects (NTDs) are sev...,Genetics & Mutations,0,0.1850,8,"http://www.ncbi.nlm.nih.gov/pubmed/23836490, h...","23836490, 24040443, 19618470, 8817329, 7958461...","[{'pmid': '23836490', 'text': 'BACKGROUND: Neu..."


In [38]:
# Explode abstracts list
df = df.explode("abstracts").reset_index(drop=True)

# Expand dictionary keys into columns
abstract_fields = pd.json_normalize(df["abstracts"])

# Merge back
df = pd.concat([df.drop(columns=["abstracts"]), abstract_fields], axis=1)

In [39]:
# Extract abstract dataframe
df_abstracts = df[["pmid", "text"]].copy()

# Drop duplicate PMIDs
df_abstracts = df_abstracts.drop_duplicates(subset=["pmid"]).reset_index(drop=True)

print("Rows after dedup:", len(df_abstracts))
print()

# Display nicely
for i, row in df_abstracts.head(5).iterrows():
    print("="*80)
    print(f"PMID: {row['pmid']}")
    print("-"*80)
    print(row["text"])
    print()

Rows after dedup: 95

PMID: 15829955
--------------------------------------------------------------------------------
text    OBJECTIVE: To analyze the prognostic factors o...
text    OBJECTIVE: To analyze the prognostic factors o...
Name: 0, dtype: object

PMID: 15617541
--------------------------------------------------------------------------------
text    OBJECTIVE: To analyze the prognostic factors o...
text    BACKGROUND: Neuromyelitis Optica Spectrum Diso...
Name: 1, dtype: object

PMID: 12239580
--------------------------------------------------------------------------------
text    OBJECTIVE: To analyze the prognostic factors o...
text    BACKGROUND: Severe forms of primary dystonia a...
Name: 2, dtype: object

PMID: 6650562
--------------------------------------------------------------------------------
text    OBJECTIVE: To analyze the prognostic factors o...
text    BACKGROUND: Recent studies linking radiation e...
Name: 3, dtype: object

PMID: 20598273
--------------------